In [ ]:
!pip install -q tokenizers transformers tensorflow matplotlib seaborn koreanize-matplotlib

In [ ]:
# ============================================================
# 실습 1-1. BPE 서브워드 토크나이저
# 목표: 텍스트가 어떻게 숫자(토큰 ID)로 변환되는지 이해한다
# 사전 설치: pip install tokenizers transformers matplotlib koreanize-matplotlib
# ============================================================

# ── 라이브러리 불러오기 ──────────────────────────────────────
from tokenizers import Tokenizer          # BPE 토크나이저 객체를 만드는 핵심 클래스
from tokenizers.models import BPE         # BPE(Byte Pair Encoding) 알고리즘 정의
from tokenizers.trainers import BpeTrainer  # BPE 학습에 필요한 설정을 담는 객체
from tokenizers.pre_tokenizers import Whitespace  # 학습 전 공백 기준으로 단어를 먼저 나눔

import koreanize_matplotlib  # matplotlib 한글 깨짐 방지 (import만 해도 자동 적용)

# ============================================================
# [실습 변수] 여기만 수정하여 다양한 실험 가능
# ============================================================

# 학습 말뭉치: AI/연구 도메인 문장 (많을수록 BPE 품질 향상)
CORPUS = [
    "자연어처리는 인공지능의 핵심 분야이다",
    "트랜스포머 모델은 어텐션 메커니즘을 사용한다",
    "연구데이터를 효율적으로 관리하는 방법이 필요하다",
    "딥러닝 기반 언어모델은 대규모 텍스트로 사전학습된다",
    "서브워드 토크나이징은 미등록어 문제를 완화한다",
    "바이트페어인코딩은 자주 등장하는 문자쌍을 병합한다",
    "어텐션 가중치는 토큰 간 관련성을 나타낸다",
    "포지셔널 인코딩은 순서 정보를 벡터에 추가한다",
    "디코더는 이전 토큰을 참고하여 다음 토큰을 예측한다",
    "온도 파라미터는 생성 텍스트의 다양성을 조절한다",
]
REPEAT = 20          # 말뭉치 반복 횟수: 적으면 BPE 학습 불안정, 많으면 시간 증가

VOCAB_SIZE = 200     # 어휘 크기: 클수록 더 많은 단어를 하나의 토큰으로 처리
                     # 너무 작으면 미등록어 [UNK] 증가, 너무 크면 학습 데이터 부족

MIN_FREQUENCY = 1    # 최소 등장 빈도: 이 횟수 이상 등장한 문자쌍만 병합
                     # 소규모 코퍼스에서는 1로 설정해야 어휘가 충분히 학습됨

# 토크나이징 테스트 단어 목록 (자유롭게 수정 가능)
TEST_WORDS = [
    "트랜스포머",          # 코퍼스에 있는 단어 -> 하나의 토큰으로 처리될 가능성 높음
    "트랜스포머모델",      # 두 단어를 붙인 합성어 -> 분절되어 처리
    "어텐션메커니즘",      # 기술 용어 결합 -> 아는 부분 + 모르는 부분 분절
    "미등록어테스트단어",  # 코퍼스에 없는 단어 -> [UNK] 또는 문자 단위 분절
]

# BPE 병합 원리 시뮬레이션 대상 단어
BPE_DEMO_WORDS = ["트랜스포머", "어텐션", "포지셔널인코딩"]

# Word-level vs BPE 비교 대상 단어
UNSEEN_WORD = "멀티헤드어텐션레이어"  # 코퍼스에 없는 긴 합성어

# ============================================================
# 이하 코드는 수정 불필요
# ============================================================

# ── 1단계: 말뭉치 파일 생성 ──────────────────────────────────
# BPE는 파일 기반으로 학습하므로 리스트를 텍스트 파일로 저장
corpus_data = CORPUS * REPEAT  # 같은 문장을 REPEAT번 반복해 빈도 확보
with open("corpus.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(corpus_data))
    # 문장 사이를 줄바꿈으로 연결해 저장
print(f"말뭉치 준비 완료: {len(corpus_data)}문장 -> corpus.txt 저장")

# ── 2단계: BPE 토크나이저 학습 ──────────────────────────────
# BPE 동작 원리:
#   1) 처음에는 모든 글자를 개별 토큰으로 시작 (예: '트','랜','스','포','머')
#   2) 가장 자주 붙어 등장하는 두 토큰을 하나로 병합 (예: '트'+'랜' -> '트랜')
#   3) VOCAB_SIZE에 도달할 때까지 반복
#   결과: 자주 쓰이는 단어/음절 조합은 하나의 토큰으로, 드문 조합은 분절

tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
# [UNK]: 어휘에 없는 문자가 등장하면 이 특수 토큰으로 대체

tokenizer.pre_tokenizer = Whitespace()
# 공백 기준으로 먼저 단어를 나눈 뒤 BPE를 적용
# 예: "트랜스포머 모델" -> ["트랜스포머", "모델"] 로 먼저 분리

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"],
    # [PAD]: 길이 맞춤 패딩, [BOS]: 문장 시작, [EOS]: 문장 끝
    show_progress=True
)

tokenizer.train(files=["corpus.txt"], trainer=trainer)
print(f"BPE 학습 완료: 어휘 크기={tokenizer.get_vocab_size()}")

# ── 3단계: 서브워드 분절 결과 확인 ───────────────────────────
print("\n" + "=" * 55)
print("BPE 서브워드 토크나이징 결과")
print("=" * 55)
print("읽는 법: 입력 단어가 몇 개의 서브워드 조각으로 나뉘는지 확인")
print("         ID는 각 조각에 할당된 정수 번호 (모델이 실제로 처리하는 값)")

for word in TEST_WORDS:
    enc = tokenizer.encode(word)
    print(f"\n입력: {word}")
    print(f"  토큰:  {enc.tokens}")
    print(f"  ID:    {enc.ids}")
    print(f"  분절 수: {len(enc.tokens)}개")

# ── 4단계: BPE 병합 원리 시뮬레이션 ──────────────────────────
print("\n" + "=" * 55)
print("BPE 병합 원리 시뮬레이션")
print("=" * 55)
print("읽는 법: '초기 문자 분리'는 BPE 시작 상태,")
print("         'BPE 적용 후'는 자주 등장한 쌍이 병합된 최종 결과")

def bpe_demo(text):
    chars = list(text)  # 문자 하나씩 분리 (BPE의 출발 상태)
    enc   = tokenizer.encode(text)  # 실제 BPE 적용 결과
    print(f"\n원본: '{text}'")
    print(f"초기 문자 분리: {chars}")
    print(f"BPE 적용 후:    {enc.tokens}")
    print(f"-> 자주 등장한 부분 문자열은 하나의 서브워드로 병합될 수 있음")

for word in BPE_DEMO_WORDS:
    bpe_demo(word)

# ── 5단계: Word-level vs BPE 비교 ────────────────────────────
print("\n" + "=" * 55)
print("Word-level vs BPE 어휘 처리 비교")
print("=" * 55)

enc = tokenizer.encode(UNSEEN_WORD)
print(f"\n미등록 합성어: '{UNSEEN_WORD}'")
print(f"Word-level -> [UNK] (단어 내부 정보 활용 어려움)")
print(f"BPE        -> {enc.tokens}  (부분 의미 보존)")
print(f"\n핵심: 현대 NLP 모델은 미등록어 대응을 위해 BPE/WordPiece 같은 서브워드 토크나이저를 사용")
print(f"      GPT 계열은 BPE, BERT 계열은 WordPiece 사용 (원리 동일)")

# ── 추가 실습: 직접 단어 입력 ────────────────────────────────
print("\n" + "=" * 55)
print("직접 테스트: 원하는 단어를 입력해 토크나이징 결과를 확인하세요")
print("=" * 55)
custom_words = [
    "자연어처리",      # 코퍼스에 있는 단어
    "멀티헤드어텐션",  # 없는 합성어
    "RAGpipeline",     # 영어+한국어 혼합
]
for w in custom_words:
    e = tokenizer.encode(w)
    print(f"  '{w}' -> {e.tokens}  (ID: {e.ids})")